In [42]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

conn = sqlite3.connect("../data/supply_chain.db")
df = pd.read_sql("SELECT * FROM supply_chain", conn)
conn.close()

print(df.shape)
df[["sku","stock_levels","order_quantities","total_lead_time","abc_class","stockout_risk"]].head()

(100, 30)


,sku,stock_levels,order_quantities,total_lead_time,abc_class,stockout_risk
0,SKU0,58,96,40,A,0
1,SKU1,53,37,62,A,0
2,SKU2,1,88,39,A,1
3,SKU3,23,59,37,A,0
4,SKU4,5,56,14,C,1


In [43]:
df["inventory_health_score"] = (
    (1 - df["defect_rates"]) * 40 +
    (1 - df["stockout_risk"]) * 30 +
    (df["availability"] / 100) * 30
).round(2)

summary = df.groupby("abc_class")[["inventory_health_score","stock_levels","defect_rates","stockout_risk"]].mean().round(3)
print(summary)

           inventory_health_score  stock_levels  defect_rates  stockout_risk
abc_class                                                                   
A                          -4.104        42.265         2.043          0.224
B                         -30.425        47.833         2.671          0.208
C                         -13.159        57.704         2.352          0.185


In [27]:
supplier = df.groupby("supplier_name").agg(
    avg_defect_rate=("defect_rates","mean"),
    avg_lead_time=("lead_times","mean"),
    total_revenue=("revenue_generated","sum"),
    stockout_count=("stockout_risk","sum"),
    sku_count=("sku","count")
).round(3).reset_index()

supplier["supplier_score"] = (
    (1 - supplier["avg_defect_rate"]) * 50 +
    (1 - supplier["avg_lead_time"] / supplier["avg_lead_time"].max()) * 30 +
    (supplier["total_revenue"] / supplier["total_revenue"].max()) * 20
).round(2)

supplier = supplier.sort_values("supplier_score", ascending=False)
print(supplier)

  supplier_name  avg_defect_rate  avg_lead_time  total_revenue  \
0    Supplier 1            1.804         16.778     157528.995   
1    Supplier 2            2.363         16.227     125467.419   
3    Supplier 4            2.337         17.000      86468.962   
2    Supplier 3            2.466         14.333      97795.980   
4    Supplier 5            2.665         14.722     110343.464   

   stockout_count  sku_count  supplier_score  
0               8         27          -19.81  
1               5         22          -50.86  
3               1         18          -55.87  
2               4         15          -56.18  
4               3         18          -65.22  


In [28]:
df["lead_time_variance"] = df["total_lead_time"] - df["total_lead_time"].mean()
df["lead_time_category"] = pd.cut(df["total_lead_time"], bins=3, labels=["Low","Medium","High"])

lt_analysis = df.groupby("lead_time_category", observed=False).agg(
    sku_count=("sku", "count"),
    avg_cost=("manufacturing_costs", "mean"),
    avg_defect=("defect_rates", "mean"),
    avg_revenue=("revenue_generated", "mean")
).round(3)
print(lt_analysis)

                    sku_count  avg_cost  avg_defect  avg_revenue
lead_time_category                                              
Low                        27    50.453       1.990     5782.149
Medium                     50    49.824       2.292     5925.001
High                       23    37.967       2.582     5445.076


In [29]:
import os
os.makedirs("../outputs", exist_ok=True)

df.to_csv("../outputs/inventory_master.csv", index=False)
supplier.to_csv("../outputs/supplier_scorecard.csv", index=False)

route_perf = df.groupby(["routes","transportation_modes"]).agg(
    avg_shipping_cost=("shipping_costs","mean"),
    avg_shipping_time=("shipping_times","mean"),
    total_revenue=("revenue_generated","sum"),
    defect_count=("defect_flag","sum")
).round(3).reset_index()
route_perf.to_csv("../outputs/route_performance.csv", index=False)

abc_summary = df.groupby(["abc_class","product_type"]).agg(
    total_revenue=("revenue_generated","sum"),
    avg_stock=("stock_levels","mean"),
    stockout_pct=("stockout_risk","mean"),
    avg_health_score=("inventory_health_score","mean")
).round(3).reset_index()
abc_summary.to_csv("../outputs/abc_summary.csv", index=False)

print("Exports done:", os.listdir("../outputs"))

Exports done: ['.ipynb_checkpoints', 'abc_summary.csv', 'abc_xyz_classification.csv', 'inventory_health.csv', 'inventory_master.csv', 'lead_time_analysis.csv', 'route_performance.csv', 'supplier_scorecard.csv']


In [30]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/supply_chain.db")

# test table exists
pd.read_sql("SELECT * FROM supply_chain_data LIMIT 5", conn)

,product_type,sku,price,availability,number_of_products_sold,revenue_generated,customer_demographics,stock_levels,lead_times,order_quantities,...,location,lead_time,production_volumes,manufacturing_lead_time,manufacturing_costs,inspection_results,defect_rates,transportation_modes,routes,costs
0,haircare,SKU0,69.808006,55,802,8661.996792,Non-binary,58,7,96,...,Mumbai,29,215,29,46.279879,Pending,0.226410,Road,Route B,187.752075
1,skincare,SKU1,14.843523,95,736,7460.900065,Female,53,30,37,...,Mumbai,23,517,30,33.616769,Pending,4.854068,Road,Route B,503.065579
2,haircare,SKU2,11.319683,34,8,9577.749626,Unknown,1,10,88,...,Mumbai,12,971,27,30.688019,Pending,4.580593,Air,Route C,141.920282
3,skincare,SKU3,61.163343,68,83,7766.836426,Non-binary,23,13,59,...,Kolkata,24,937,18,35.624741,Fail,4.746649,Rail,Route A,254.776159
4,skincare,SKU4,4.805496,26,871,2686.505152,Non-binary,5,3,56,...,Delhi,5,414,3,92.065161,Fail,3.145580,Air,Route A,923.440632


In [31]:
with open("../sql/02_inventory_health.sql", "r") as f:
    query = f.read()

df = pd.read_sql(query, conn)

df.head()

,product_type,sku,total_stock,total_orders,avg_lead_time,avg_defect_rate,stock_to_demand_ratio,inventory_status
0,haircare,SKU74,41,1,1.0,0.396613,41.00,Overstocked
1,haircare,SKU46,92,6,18.0,3.648611,15.33,Overstocked
2,haircare,SKU97,46,4,10.0,3.376238,11.50,Overstocked
3,cosmetics,SKU49,97,9,28.0,2.825814,10.78,Overstocked
4,cosmetics,SKU21,69,7,19.0,0.018608,9.86,Overstocked


In [32]:
with open("../sql/03_supplier_scorecard.sql", "r") as f:
    query = f.read()

df_supplier = pd.read_sql(query, conn)

df_supplier.head()

,supplier_name,total_skus,avg_lead_time,avg_defect_rate,avg_shipping_cost,total_revenue,supplier_performance
0,Supplier 1,27,14.78,1.80,5.51,157529.00,Top Supplier
1,Supplier 2,22,18.55,2.36,5.74,125467.42,Average Supplier
2,Supplier 5,18,18.06,2.67,5.79,110343.46,Average Supplier
3,Supplier 3,15,20.13,2.47,4.79,97795.98,Average Supplier
4,Supplier 4,18,15.22,2.34,5.76,86468.96,Average Supplier


In [33]:
df_supplier.to_csv("../outputs/supplier_scorecard.csv", index=False)

In [34]:
with open("../sql/02_inventory_health.sql", "r") as f:
    query_inventory = f.read()

df_inventory = pd.read_sql(query_inventory, conn)

df_inventory.head()

,product_type,sku,total_stock,total_orders,avg_lead_time,avg_defect_rate,stock_to_demand_ratio,inventory_status
0,haircare,SKU74,41,1,1.0,0.396613,41.00,Overstocked
1,haircare,SKU46,92,6,18.0,3.648611,15.33,Overstocked
2,haircare,SKU97,46,4,10.0,3.376238,11.50,Overstocked
3,cosmetics,SKU49,97,9,28.0,2.825814,10.78,Overstocked
4,cosmetics,SKU21,69,7,19.0,0.018608,9.86,Overstocked


In [35]:
df_inventory.to_csv("../outputs/inventory_health.csv", index=False)

In [36]:
df_supplier.to_csv("../outputs/supplier_scorecard.csv", index=False)

print("Supplier scorecard exported successfully!")

Supplier scorecard exported successfully!


In [37]:
with open("../sql/04_lead_time_analysis.sql", "r") as f:
    query = f.read()

df_lead = pd.read_sql(query, conn)

df_lead.head()

,supplier_name,avg_lead_time,min_lead_time,max_lead_time,lead_time_variability,total_orders,risk_category
0,Supplier 4,15.22,1.0,29.0,28.0,18,High Risk
1,Supplier 2,18.55,2.0,30.0,28.0,22,High Risk
2,Supplier 1,14.78,1.0,29.0,28.0,27,High Risk
3,Supplier 5,18.06,1.0,28.0,27.0,18,High Risk
4,Supplier 3,20.13,5.0,30.0,25.0,15,High Risk


In [38]:
with open("../sql/04_lead_time_analysis.sql", "r") as f:
    query = f.read()

df_lead = pd.read_sql(query, conn)

df_lead.head()

,supplier_name,avg_lead_time,min_lead_time,max_lead_time,lead_time_variability,total_orders,risk_category
0,Supplier 4,15.22,1.0,29.0,28.0,18,High Risk
1,Supplier 2,18.55,2.0,30.0,28.0,22,High Risk
2,Supplier 1,14.78,1.0,29.0,28.0,27,High Risk
3,Supplier 5,18.06,1.0,28.0,27.0,18,High Risk
4,Supplier 3,20.13,5.0,30.0,25.0,15,High Risk


In [39]:
df_lead.to_csv("../outputs/lead_time_analysis.csv", index=False)

print("Lead time analysis exported!")

Lead time analysis exported!


In [40]:
with open("../sql/05_abc_xyz_classification.sql", "r") as f:
    query = f.read()

df_abc = pd.read_sql(query, conn)

df_abc.head()

,sku,product_type,total_revenue,avg_demand,abc_category,demand_category,abc_xyz_class
0,SKU51,haircare,9866.465458,52.0,A,Y,AY
1,SKU38,cosmetics,9692.318040,88.0,A,X,AX
2,SKU31,skincare,9655.135103,44.0,A,Y,AY
3,SKU90,skincare,9592.633570,96.0,A,X,AX
4,SKU2,haircare,9577.749626,88.0,A,X,AX


In [41]:
df_abc.to_csv("../outputs/abc_xyz_classification.csv", index=False)

print("ABC XYZ classification exported!")

ABC XYZ classification exported!
